# Sentinel-X — YOLO (people) + OpenCV (face & eyes)

Run the cells **top to bottom**.

| Who | Does what | Drawn as |
|-----|-----------|----------|
| **YOLO** | detects and tracks every *person* | **orange square** |
| **OpenCV** (Haar cascades) | detects the *face* and the *eyes* inside each person's head area | **green square** + **green dots** on the eyes |

Keys in the video window: **q** or **X** = quit, **b** = orange boxes on/off. Nothing is saved.

## Part 1 — Install (once)

`ultralytics` provides YOLO (installs PyTorch), `opencv-python` reads the webcam, detects faces/eyes and draws.
MediaPipe is **no longer needed**.

In [1]:
%pip install -U ultralytics opencv-python
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
%pip install "opencv-python<5"

  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.14.0.94
    Uninstalling opencv-python-4.14.0.94:
      Successfully uninstalled opencv-python-4.14.0.94
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\grym\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Note: you may need to restart the kernel to use updated packages.


  Using cached opencv_python-4.14.0.94-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.14.0.94-cp37-abi3-win_amd64.whl (41.2 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\grym\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:
import os, sys, time, cv2

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)

# 1) Cascade check
path = cv2.data.haarcascades
for f in ["haarcascade_frontalface_default.xml", "haarcascade_eye.xml"]:
    full = os.path.join(path, f)
    exists = os.path.isfile(full)
    loaded = exists and not cv2.CascadeClassifier(full).empty()
    print(f"{f}: exists={exists}, loads={loaded}")

# 2) Camera check: which index/backend gives a non-black frame?
names = {cv2.CAP_DSHOW: "DSHOW", cv2.CAP_MSMF: "MSMF", cv2.CAP_ANY: "ANY"}
for idx in range(4):
    for backend, name in names.items():
        cap = cv2.VideoCapture(idx, backend)
        if not cap.isOpened():
            cap.release()
            continue
        best = None
        t0 = time.time()
        while time.time() - t0 < 2:
            ok, frame = cap.read()
            if ok and frame is not None:
                best = frame.mean()
                if best > 8:
                    break
            time.sleep(0.03)
        cap.release()
        print(f"index {idx} / {name}: opened, brightness={best if best is None else round(best, 1)}"
              f"{'  <-- WORKS' if best and best > 8 else ''}")

Python: 3.14.6
OpenCV: 4.14.0
haarcascade_frontalface_default.xml: exists=True, loads=True
haarcascade_eye.xml: exists=True, loads=True
index 0 / DSHOW: opened, brightness=68.3  <-- WORKS
index 0 / MSMF: opened, brightness=107.4  <-- WORKS
index 0 / ANY: opened, brightness=119.3  <-- WORKS


## Part 2 — Settings

If you change something, re-run **Parts 2 → 7**.

- `MODEL_NAME` / `YOLO_IMGSZ` / `PERSON_CONF`: YOLO person detector.
- `HEAD_FRAC`: top part of the person box searched for a face.
- `FACE_SCALE` / `FACE_NEIGHBORS` / `FACE_MIN_PX`: OpenCV face detector (more neighbors = fewer false positives).
- `EYE_NEIGHBORS`: OpenCV eye detector.
- `FACES_PER_FRAME` / `FACE_EVERY`: how many people are scanned per frame / min frames between two scans.

In [4]:
# --- camera / display ---
SOURCE = 0                      # 0 = default webcam, 1 = second camera, or a video file path
CAM_W, CAM_H = 1920, 1080       # requested camera resolution
WIDTH  = 1280                   # width of the displayed window
WINDOW = "Sentinel-X camera (q or X to quit)"

# --- YOLO person detector (orange square) ---
MODEL_NAME     = "yolo26s.pt"   # yolo26n / s / m / l  (downloaded automatically the first time)
MODEL_FALLBACK = "yolo11s.pt"   # used if MODEL_NAME cannot be loaded
YOLO_IMGSZ     = 1280           # multiple of 32
PERSON_CONF    = 0.25
TRACKER        = "bytetrack.yaml"
DEVICE         = None           # None = automatic (GPU if available), or "cpu", "0", "mps"

# --- OpenCV face + eyes (green square + green dots) ---
HEAD_FRAC       = 0.4           # top 40 % of the person box is searched for a face
FACE_MIN_SIDE   = 256           # small head crops are enlarged to at least this many px
FACE_SCALE      = 1.1           # Haar scaleFactor
FACE_NEIGHBORS  = 5             # higher = fewer false faces
FACE_MIN_PX     = 40            # smallest face (in the enlarged crop)
EYE_NEIGHBORS   = 6             # higher = fewer false eyes
FACES_PER_FRAME = 3             # max face scans per frame
FACE_EVERY      = 2             # min frames between two scans of the same person

## Part 3 — Imports and OpenCV cascades

The Haar cascade files ship with `opencv-python`, nothing to download.

In [5]:
import time

import cv2
import numpy as np
from ultralytics import YOLO

FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
EYE_CASCADE  = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")
assert not FACE_CASCADE.empty() and not EYE_CASCADE.empty(), "OpenCV cascades not found"
print("OpenCV", cv2.__version__, "- face & eye cascades ready")

OpenCV 4.14.0 - face & eye cascades ready


## Part 4 — Helpers

- `describe_device`: tells you whether YOLO will run on a GPU or the CPU.
- `open_camera`: tries several OpenCV backends until a real frame comes out.

In [6]:
def describe_device():
    if DEVICE is not None:
        return f"device forced to {DEVICE}"
    try:
        import torch
        if torch.cuda.is_available():
            return "GPU: " + torch.cuda.get_device_name(0)
        return "CPU (no CUDA GPU found)"
    except Exception:
        return "device unknown"


def open_camera():
    """DSHOW only (the only backend that works here). Default resolution first, reject black frames."""
    for (w, h) in ((None, None), (1280, 720), (1920, 1080)):
        for use_mjpg in (False, True):
            cap = cv2.VideoCapture(SOURCE, cv2.CAP_DSHOW)
            if not cap.isOpened():
                cap.release()
                continue
            if use_mjpg:
                cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
            if w:
                cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
                cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
            t0 = time.time()
            while time.time() - t0 < 3:              # let auto-exposure settle
                ok, frame = cap.read()
                if ok and frame is not None and frame.mean() > 8:
                    print(f"Camera OK: requested={w or 'default'}, MJPG={use_mjpg}, "
                          f"actual={frame.shape[1]}x{frame.shape[0]}, mean={frame.mean():.0f}")
                    return cap
                time.sleep(0.03)
            cap.release()
    return None

## Part 5 — The detection pipeline

For every frame:

1. **YOLO** (with tracking) returns the people boxes (orange) and a stable ID per person.
2. For each person, the **head area** (top `HEAD_FRAC` of the box) is cropped and enlarged if small. **OpenCV** looks for a face (green square), then for eyes inside the upper part of that face (green dots).
3. Face and eyes are stored **relative to the head area**, so between two scans they follow the person's box smoothly.

OpenCV's Haar cascades work best on faces that look at the camera.

In [7]:
class PersonFacePipeline:

    def __init__(self):
        self.person = self._load_yolo()
        self.show_boxes = True                            # key b : orange person boxes
        self.use_track = True
        self.states = {}                                  # per person: face/eyes, last scan...
        self.frame_i = 0

    # ---- internals -------------------------------------------------------
    @staticmethod
    def _load_yolo():
        try:
            return YOLO(MODEL_NAME)
        except Exception as e:
            print(f"Could not load {MODEL_NAME} ({e}).")
            print(f"Falling back to {MODEL_FALLBACK}.")
            return YOLO(MODEL_FALLBACK)

    def _run_yolo(self, frame):
        kw = dict(classes=[0], conf=PERSON_CONF, imgsz=YOLO_IMGSZ, verbose=False)   # class 0 = person
        if DEVICE is not None:
            kw["device"] = DEVICE
        if self.use_track:
            try:
                return self.person.track(frame, persist=True, tracker=TRACKER, **kw)[0]
            except Exception as e:
                print("Tracking unavailable, using plain detection:", e)
                self.use_track = False
        return self.person.predict(frame, **kw)[0]

    @staticmethod
    def _head_region(box, w, h):
        x1, y1, x2, y2 = box
        bw, bh = x2 - x1, y2 - y1
        rx0, rx1 = max(0, int(x1 - 0.1 * bw)), min(w, int(x2 + 0.1 * bw))
        ry0, ry1 = max(0, int(y1 - 0.05 * bh)), min(h, int(y1 + HEAD_FRAC * bh))
        return rx0, ry0, rx1, ry1

    def _scan_face(self, gray, p, w, h):
        """OpenCV: face in the head area, then eyes in the face."""
        st = p["state"]
        st["last_scan"] = self.frame_i
        rx0, ry0, rx1, ry1 = self._head_region(p["box"], w, h)
        cw, ch = rx1 - rx0, ry1 - ry0
        if cw < 16 or ch < 16:
            return
        crop = gray[ry0:ry1, rx0:rx1]
        s = max(1.0, FACE_MIN_SIDE / max(cw, ch))         # enlarge small (far) heads
        if s > 1:
            crop = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        crop = cv2.equalizeHist(np.ascontiguousarray(crop))
        H, W = crop.shape[:2]

        faces = FACE_CASCADE.detectMultiScale(crop, scaleFactor=FACE_SCALE,
                                              minNeighbors=FACE_NEIGHBORS,
                                              minSize=(FACE_MIN_PX, FACE_MIN_PX))
        if len(faces) == 0:
            st["miss"] += 1
            if st["miss"] >= 3:                           # face lost for 3 scans -> forget it
                st["face"], st["eyes"] = None, []
            return

        fx, fy, fw, fh = max(faces, key=lambda f: f[2] * f[3])    # biggest face
        st["miss"] = 0
        st["face"] = (fx / W, fy / H, (fx + fw) / W, (fy + fh) / H)   # relative to head area

        # eyes: only in the upper 60 % of the face
        roi = crop[fy:fy + int(0.6 * fh), fx:fx + fw]
        eyes = EYE_CASCADE.detectMultiScale(roi, scaleFactor=1.1, minNeighbors=EYE_NEIGHBORS,
                                            minSize=(max(8, fw // 10), max(8, fw // 10)))
        eyes = sorted(eyes, key=lambda e: e[2] * e[3], reverse=True)[:2]   # 2 biggest
        st["eyes"] = [((fx + ex + ew / 2) / W, (fy + ey + eh / 2) / H) for ex, ey, ew, eh in eyes]

    # ---- public ----------------------------------------------------------
    def detect(self, frame):
        """frame: full-resolution BGR image -> list of people (dicts with box, id, conf, face, eyes)."""
        h, w = frame.shape[:2]
        self.frame_i += 1
        if not self.use_track:
            self.states = {}

        # 1) YOLO: persons
        res = self._run_yolo(frame)
        people = []
        boxes = res.boxes
        if boxes is not None and len(boxes):
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            if self.use_track and boxes.id is not None:
                ids = boxes.id.int().cpu().tolist()
            else:
                ids = [None] * len(xyxy)
            for i, (b, c, tid) in enumerate(zip(xyxy, confs, ids)):
                key = tid if tid is not None else f"n{i}"
                st = self.states.setdefault(key, {"face": None, "eyes": [], "last_scan": -10**9,
                                                  "miss": 0, "seen": 0})
                st["seen"] = self.frame_i
                people.append({"key": key, "id": tid, "box": tuple(float(v) for v in b),
                               "conf": float(c), "state": st, "face": None, "eyes": []})

        # 2) OpenCV: face + eyes (people not scanned for the longest time first)
        due = [p for p in people if self.frame_i - p["state"]["last_scan"] >= FACE_EVERY]
        due.sort(key=lambda p: p["state"]["last_scan"])
        if due:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            for p in due[:FACES_PER_FRAME]:
                self._scan_face(gray, p, w, h)

        # 3) re-project the stored face/eyes onto each person's current head area
        for p in people:
            st = p["state"]
            if st["face"]:
                rx0, ry0, rx1, ry1 = self._head_region(p["box"], w, h)
                cw, ch = rx1 - rx0, ry1 - ry0
                u0, v0, u1, v1 = st["face"]
                p["face"] = (int(rx0 + u0 * cw), int(ry0 + v0 * ch),
                             int(rx0 + u1 * cw), int(ry0 + v1 * ch))
                p["eyes"] = [(int(rx0 + u * cw), int(ry0 + v * ch)) for u, v in st["eyes"]]

        self.states = {k: v for k, v in self.states.items() if self.frame_i - v["seen"] <= 30}
        return people

    def warmup(self, w, h):
        """Run YOLO once on a blank image so the first real frames are not slow."""
        dummy = np.zeros((h, w, 3), np.uint8)
        for _ in range(2):
            self.detect(dummy)
        self.states = {}
        self.frame_i = 0

    def draw(self, view, people, scale):
        ORANGE, GREEN = (0, 165, 255), (0, 255, 0)        # BGR
        n_faces = 0
        for p in people:
            # orange square = person (YOLO)
            if self.show_boxes:
                x1, y1, x2, y2 = [int(v * scale) for v in p["box"]]
                cv2.rectangle(view, (x1, y1), (x2, y2), ORANGE, 2)
                cv2.putText(view, f"Personne {p['conf'] * 100:.0f}%", (x1, max(y1 - 6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, ORANGE, 1, cv2.LINE_AA)

            if p["face"]:
                n_faces += 1
                fx0, fy0, fx1, fy1 = [int(v * scale) for v in p["face"]]
                # green square = face (OpenCV)
                cx, cy = (fx0 + fx1) // 2, (fy0 + fy1) // 2
                half = max(fx1 - fx0, fy1 - fy0) // 2
                cv2.rectangle(view, (cx - half, cy - half), (cx + half, cy + half), GREEN, 2)
                # green dots = eyes (OpenCV)
                for ex, ey in p["eyes"]:
                    cv2.circle(view, (int(ex * scale), int(ey * scale)), 3, GREEN, -1, cv2.LINE_AA)

        cv2.putText(view, f"people: {len(people)}  faces: {n_faces}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, GREEN, 2)

    def on_key(self, key):
        if key == ord("b"):
            self.show_boxes = not self.show_boxes

    def close(self):
        pass

## Part 6 — Camera loop

`show_feed(pipeline)` opens the camera, runs the pipeline on every full-resolution frame, then resizes **only the copy shown on screen**. This cell only *defines* the function.

In [8]:
def window_closed():
    """True once the user clicked the X in the window's title bar."""
    try:
        visible = cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE)
        autosize = cv2.getWindowProperty(WINDOW, cv2.WND_PROP_AUTOSIZE)
        return visible < 1 or autosize < 0
    except cv2.error:
        return True


def show_feed(pipeline):
    cap = open_camera()
    if cap is None:
        print("No working camera. Close apps using it (Teams, Zoom, browser), "
              "check Windows camera privacy settings, or try SOURCE = 1.")
        return

    real_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    real_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Camera opened at {real_w}x{real_h} (requested {CAM_W}x{CAM_H}).")

    n, fps, t_prev = 0, 0.0, time.perf_counter()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("No frame received (camera lost or video ended).")
                break
            n += 1

            people = pipeline.detect(frame)                          # YOLO + OpenCV, full resolution

            h, w = frame.shape[:2]
            scale = WIDTH / w
            view = cv2.resize(frame, (WIDTH, int(h * scale)))        # display copy only
            pipeline.draw(view, people, scale)

            t_now = time.perf_counter()
            fps = 0.9 * fps + 0.1 / max(t_now - t_prev, 1e-6) if n > 1 else 0.0
            t_prev = t_now
            cv2.putText(view, f"{fps:.0f} fps", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.imshow(WINDOW, view)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            if key != 255:
                pipeline.on_key(key)
            if n > 2 and window_closed():
                break
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        print(f"Camera released. {n} frames shown.")

## Part 7 — Run

1. Load and warm up **YOLO**.
2. **Then** open the camera with **OpenCV**: orange square = person, green square + green dots = face and eyes.
3. `q` or the window's X quits, `b` toggles the orange boxes.

In [9]:
print("1/3 Loading YOLO...")
pipeline = PersonFacePipeline()

print("2/3 Warming up YOLO...")
pipeline.warmup(CAM_W, CAM_H)

print(f"3/3 YOLO ready ({describe_device()}). Opening the camera with OpenCV now.")
try:
    show_feed(pipeline)          # q or X = quit, b = orange boxes
finally:
    pipeline.close()

1/3 Loading YOLO...
2/3 Warming up YOLO...
3/3 YOLO ready (GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU). Opening the camera with OpenCV now.
Camera OK: requested=default, MJPG=False, actual=640x480, mean=119
Camera opened at 640x480 (requested 1920x1080).
Camera released. 675 frames shown.


## Tuning guide

| Problem | Change (Part 2) |
|---------|-----------------|
| People missed far away | bigger model (`yolo26m.pt`), higher `YOLO_IMGSZ`, lower `PERSON_CONF` (0.15) |
| FPS too low | `yolo26n.pt`, lower `YOLO_IMGSZ`, lower `FACES_PER_FRAME`, higher `FACE_EVERY` |
| Face not found | face must look at the camera; try `FACE_NEIGHBORS = 3`, `HEAD_FRAC = 0.5` |
| False faces | raise `FACE_NEIGHBORS` (6-8) |
| Missing / extra eye dots | lower / raise `EYE_NEIGHBORS` |